# Dataset agreement and validation, 2013–2025

This is the single active `00_` analysis notebook. It combines the former general-sanity/cleanliness and validation notebooks, whose original versions are preserved in `_archived/`.

**Part A** analyses saved three-model candidate labels: consensus, pairwise agreement, annual trends, explicit UK Biobank mentions, keyword profiles, TF-IDF terms and optional semantic separation. **Part B** evaluates the six prompt strategies and encoder baselines against labelled validation papers and produces performance and agreement figures alongside ranking tables.

All records are restricted to **1 January 2013–31 December 2025 inclusive**. Missing inputs are reported as **SKIP**, independently for each section; a skip does not mean the scientific checks passed. Invalid data are reported as **FAIL** and cause the notebook to fail after both sections have been attempted. No tagging models are needed for Part A. New semantic encoding and validation inference are disabled by default; matching saved results are reused.

Figures use Helvetica, the shared palette, bold uppercase left-aligned titles, and 500-DPI PNG/PDF exports.

### Publication figure set

Related diagnostics are combined into four figures, plus an optional semantic figure:

| Export prefix | Panels | Source tables |
|---|---|---|
| `00_01_figure_candidate_agreement` | A–C: model prediction/parsing rates, TRUE votes and pairwise agreement; D–F: annual candidate counts, consensus rate and model predictions | `model_positive_rate_summary.csv`, `true_vote_distribution.csv`, `pairwise_model_agreement.csv`, `yearly_three_model_consensus_trend.csv` |
| `00_02_figure_candidate_text` | A: mention and keyword profiles; B,C: strongest TF-IDF contrasts in each direction | `keyword_category_summary.csv`, `tfidf_discriminative_terms.csv` |
| `00_03_figure_validation_performance` | Accuracy, precision, recall and F1 across models and prompts | `ALL_prompt_model_results_summary.csv` |
| `00_04_figure_validation_agreement` | Pairwise model agreement for each of six prompt strategies | `pairwise_agreement_percent_*.csv`, `pairwise_agreement_n_*.csv` |
| `00_05_figure_semantic_diagnostics` (optional) | The same semantic projection coloured by consensus group and publication year | `semantic_sample_with_coordinates.csv`, `semantic_metrics.csv` |

Each figure has a matching `_caption.txt` for the manuscript. Heatmaps use the shared blue–cream–red palette with cell outlines and explicit scales. Unanimous TRUE candidates are red and other candidates blue throughout their comparisons. Redundant standalone figures are no longer produced; their full numerical tables remain available. A figure is generated only when its source data are available.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()


## 1. Inputs and run options

Part A automatically reads `data/analysis/dataset/matched_ukb_full_final_2013_2025_three_model_labels.csv`. No path configuration is needed. Set `UKB_COMBINED_LABELS_CSV` to override this with another **full candidate-level** saved three-model CSV. If the default is absent, recognised current or legacy label filenames under `data/` are detected when unique. Showcase+ and the TRUE-only export cannot replace this input.

For new Part B inference, set `UKB_VALIDATION_POSITIVE_CSV`, `UKB_VALIDATION_NEGATIVE_CSV`, `UKB_VALIDATION_N_POS` and `UKB_VALIDATION_N_NEG`. Defaults are `data/validation/ukb_ground_truth_positive_labelled.csv` and `data/validation/ukb_negative_pre2014_labelled.csv`. Both classes need enough in-window records for evaluation and separate prompt examples. The historical `negative_pre2014` filename does not waive the date restriction: only eligible 2013 records can enter from that file.

Saved per-paper predictions in `VALIDATION_OUTPUT_DIR` are checked before model imports. New validation inference requires `RUN_VALIDATION_INFERENCE = True`; it can download large model weights. The optional semantic diagnostic similarly requires `RUN_SEMANTIC_ANALYSIS = True` when no matching coordinates and metrics are available.

In [ ]:
RUN_AGREEMENT = True
RUN_VALIDATION = True
RUN_SEMANTIC_ANALYSIS = os.environ.get("UKB_RUN_SEMANTIC_ANALYSIS", "0") == "1"
RUN_VALIDATION_INFERENCE = os.environ.get("UKB_RUN_VALIDATION_INFERENCE", "0") == "1"

COMBINED_LABELS_CSV = os.environ.get("UKB_COMBINED_LABELS_CSV", "").strip() or None
VALIDATION_POSITIVE_CSV = os.environ.get("UKB_VALIDATION_POSITIVE_CSV", "").strip() or None
VALIDATION_NEGATIVE_CSV = os.environ.get("UKB_VALIDATION_NEGATIVE_CSV", "").strip() or None
VALIDATION_N_POS = os.environ.get("UKB_VALIDATION_N_POS", "").strip() or None
VALIDATION_N_NEG = os.environ.get("UKB_VALIDATION_N_NEG", "").strip() or None
VALIDATION_OUTPUT_DIR = Path(os.environ.get("UKB_VALIDATION_OUTPUT_DIR", "").strip() or P.OUTPUT / "validation").expanduser()
if not VALIDATION_OUTPUT_DIR.is_absolute():
    VALIDATION_OUTPUT_DIR = P.ROOT / VALIDATION_OUTPUT_DIR

TABLE_DIR = P.TABLE_DATA_ANALYSIS / "00_dataset"
FIGURE_DIR = P.FIG_DATA_ANALYSIS / "00_dataset"
MAX_TFIDF_PER_GROUP = 20_000
MAX_SEMANTIC_PER_GROUP = 3_000
SEED = 42


In [ ]:
import pandas as pd
from IPython.display import display
from utils.shared_style import load_style
from utils import data_analysis_00_agreement as A
from utils import data_analysis_00_validation as V

STYLE = load_style("00_dataset")
section_results = []

def run_section(name, enabled, function, **kwargs):
    if not enabled:
        result = {"section": name, "status": "SKIP", "detail": "Disabled in notebook configuration."}
        print(f"[SKIP] {name}: disabled in configuration.")
    else:
        try:
            result = function(**kwargs)
            result.setdefault("section", name)
            result.setdefault("detail", result.get("reason", "Completed."))
        except Exception as error:
            detail = f"{type(error).__name__}: {str(error).splitlines()[0]}"
            print(f"[FAIL] {name}: {detail}")
            result = {"section": name, "status": "FAIL", "detail": detail}
    section_results.append(result)
    return result


## 2. Saved three-model agreement and candidate characteristics

All calculations read existing Qwen, Llama and Mistral labels; these classifiers are never rerun here. Agreement and annual trends are combined in `00_01`; mention/keyword profiles and directional TF-IDF contrasts are combined in `00_02`. All detailed tables, including consensus categories and complete term scores, are retained. Tables go to `output/tables/data_analysis/00_dataset/three_model_agreement/` and figures to the corresponding `output/figures/` directory.

The optional `00_05` places the two semantic maps side by side. Matching semantic coordinates and metrics are reused after checking sampled IDs, texts, years and groups. A missing semantic cache skips only that diagnostic unless explicitly enabled. Set `show_tables=True` below to also display the detailed tables inline.

In [ ]:
agreement_result = run_section(
    "Three-model agreement", RUN_AGREEMENT, A.run_agreement,
    input_path=COMBINED_LABELS_CSV,
    output_dir=TABLE_DIR / "three_model_agreement",
    figure_dir=FIGURE_DIR / "three_model_agreement",
    run_semantic=RUN_SEMANTIC_ANALYSIS,
    max_tfidf_per_group=MAX_TFIDF_PER_GROUP,
    max_semantic_per_group=MAX_SEMANTIC_PER_GROUP,
    seed=SEED, show_tables=False, show_figures=True,
)


## 3. Labelled validation across six prompt strategies

Completed, consistent per-paper predictions are reused to reconstruct performance metrics, model rankings and two combined figures: a four-panel performance comparison (`00_03`) and six-panel agreement comparison (`00_04`). Encoder rows are visibly separated and marked as in-sample calibration results. Partial or incompatible caches raise an error instead of silently starting new models. Fresh inference retains the original prompt/model workflow and runs only when explicitly enabled.

Validation tables remain under `VALIDATION_OUTPUT_DIR` (by default `output/validation/`); figures are collected under `output/figures/data_analysis/00_dataset/validation/`.

**Interpretation:** the original SciBERT and S-BERT baselines optimise their similarity thresholds using the evaluation labels. Their performance is an in-sample calibration result, not an independent held-out estimate. Pairwise agreement measures consistency, not accuracy.

In [ ]:
validation_result = run_section(
    "Labelled validation", RUN_VALIDATION, V.run_validation,
    output_dir=VALIDATION_OUTPUT_DIR,
    positive_csv=VALIDATION_POSITIVE_CSV,
    negative_csv=VALIDATION_NEGATIVE_CSV,
    n_positive=VALIDATION_N_POS, n_negative=VALIDATION_N_NEG,
    run_inference=RUN_VALIDATION_INFERENCE,
    figure_dir=FIGURE_DIR / "validation",
)


## 4. Completion status

This table distinguishes completed analyses from missing or disabled sections. It is saved even when inputs are absent. Any analysis failure is raised after both sections have been attempted.

In [ ]:
summary = pd.DataFrame([
    {"section": result["section"], "status": result["status"], "detail": result["detail"]}
    for result in section_results
])
if agreement_result.get("semantic_status") == "SKIP":
    summary.loc[len(summary)] = ["Semantic separation", "SKIP", "No matching semantic cache; encoding disabled."]
TABLE_DIR.mkdir(parents=True, exist_ok=True)
status_path = TABLE_DIR / "00_dataset_status.csv"
summary.to_csv(status_path, index=False)
display(summary)
counts = summary["status"].value_counts()
print(f"{counts.get('PASS', 0)} completed, {counts.get('SKIP', 0)} skipped, {counts.get('FAIL', 0)} failed.")
print("Status:", P.raw_path(status_path))
if summary["status"].eq("FAIL").any():
    raise RuntimeError("Dataset analysis failed: " + "; ".join(summary.loc[summary["status"].eq("FAIL"), "detail"]))
